# Gradient Boosting — eenvoudige fold-lokale voorbereiding

**Concept op verzoek van de gebruiker, uitgewerkt met Codex.** Milan moet
de keuzes en eigen bijdrage zelf beoordelen. Alle nieuwe bestanden blijven onder
`solutions/milan`. We gebruiken de cursusmethoden en de oorspronkelijke elf
invoervelden. Geen nieuwe onafhankelijke eindtest of menselijke review geclaimd.
De dure training is afzonderlijk werkelijk uitgevoerd; onderstaande rapportcellen
    worden van boven naar beneden uitgevoerd; de precieze uitvoeringsmethode staat
in de notebookmetadata en uitvoeringsreceipt.

## Model en zoekruimte
We gebruiken exact dezelfde buitenfolds. Per familie selecteren drie binnenfolds de instellingen. Scaling wordt bij SVM binnen de pipeline geleerd; boommodellen krijgen geen onnodige standaardisatie.

In [1]:
from pathlib import Path
import json, sys
import numpy as np
import pandas as pd
HERE = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'results_v2/summary.csv').exists())
sys.path.insert(0, str(HERE))
from mushroom_alternative import development_data, scores
from improve import experiments, make_pipeline, Measurements
X, y, ids, folds = development_data()
summary = pd.read_csv(HERE/'results_v2/summary.csv')
details = pd.read_csv(HERE/'results_v2/outer_fold_metrics.csv')
selections = json.loads((HERE/'results_v2/selections.json').read_text())
families = ['gradient_boosting']
for family in families:
    estimator, grid = experiments()[family]
    display(estimator)
    print('Zoekruimte:', grid)


Zoekruimte: [{'preprocessor': [ColumnTransformer(sparse_threshold=0,
                  transformers=[('num',
                                 Pipeline(steps=[('impute',
                                                  SimpleImputer(add_indicator=True,
                                                                strategy='median'))]),
                                 ['cap_diameter', 'stem_height', 'stem_width']),
                                ('cat',
                                 Pipeline(steps=[('impute',
                                                  SimpleImputer(fill_value='missing',
                                                                strategy='constant')),
                                                 ('encode',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                sparse_output=False))]),
                                 ['spore_print_color', 'gill_color',

Pipeline(steps=[('measurements', Measurements()),
                ('preprocessor',
                 ColumnTransformer(sparse_threshold=0,
                                   transformers=[('num',
                                                  Pipeline(steps=[('impute',
                                                                   SimpleImputer(add_indicator=True,
                                                                                 strategy='median'))]),
                                                  ['cap_diameter',
                                                   'stem_height',
                                                   'stem_width']),
                                                 ('cat',
                                                  Pipeline(steps=[('impute',
                                                                   SimpleImputer(fill_value='missing',
                                                                                 strategy='constant')),
                                                                  ('encode',
                                                                   OneHotEncoder(handle_unknown='ignore',
                                                                                 sparse_output=False))]),
                                                  ['spore_print_color',
                                                   'gill_color', 'habitat',
                                                   'season', 'ring_type',
                                                   'cap_shape', 'stem_surface',
                                                   'jumbled_noise_1'])])),
                ('classifier',
                 GradientBoostingClassifier(min_samples_leaf=3,
                                            n_estimators=300, random_state=42,
                                            subsample=0.8))])

## De werkelijk uitgevoerde selectie
Deze tabel geeft per buitenfold de binnen training gekozen instellingen en drempel. Een binnenscore is een selectiescore, niet de buitenmeting.

In [2]:
display(pd.DataFrame([s for s in selections if s['family'] in families]))
for family in families:
    display(details[details.model.str.startswith(family+'_')])


,family,fold,best_parameters,inner_cv_macro_f1,f1_threshold,seconds
0,gradient_boosting,1,"{'classifier__learning_rate': 0.05, 'classifie...",0.748730,0.48,58.395397
1,gradient_boosting,2,"{'classifier__learning_rate': 0.05, 'classifie...",0.749036,0.49,64.901603
2,gradient_boosting,3,"{'classifier__learning_rate': 0.05, 'classifie...",0.760887,0.43,66.775231
3,gradient_boosting,4,"{'classifier__learning_rate': 0.05, 'classifie...",0.732405,0.41,66.183278
4,gradient_boosting,5,"{'classifier__learning_rate': 0.05, 'classifie...",0.740791,0.42,63.784116


,model,fold,threshold,accuracy,macro_f1,balanced_accuracy,recall_p,precision_p,specificity_e,roc_auc,average_precision,tn,fp,fn,tp
15,gradient_boosting_default,1,0.50,0.780488,0.754236,0.745008,0.598071,0.771784,0.891945,0.796948,0.731772,454,55,125,186
16,gradient_boosting_f1_threshold,1,0.48,0.779268,0.753527,0.744651,0.601286,0.766393,0.888016,0.796948,0.731772,452,57,124,187
17,gradient_boosting_default,2,0.50,0.786585,0.757729,0.746792,0.581994,0.800885,0.911591,0.819752,0.746909,464,45,130,181
18,gradient_boosting_f1_threshold,2,0.49,0.785366,0.758830,0.748937,0.598071,0.784810,0.899804,0.819752,0.746909,458,51,125,186
19,gradient_boosting_default,3,0.50,0.784146,0.758974,0.749937,0.609677,0.771429,0.890196,0.824042,0.758578,454,56,121,189
20,gradient_boosting_f1_threshold,3,0.43,0.787805,0.769341,0.764263,0.667742,0.744604,0.860784,0.824042,0.758578,439,71,103,207
21,gradient_boosting_default,4,0.50,0.817073,0.797319,0.787793,0.667742,0.814961,0.907843,0.842328,0.795417,463,47,103,207
22,gradient_boosting_f1_threshold,4,0.41,0.786585,0.772060,0.770873,0.706452,0.722772,0.835294,0.842328,0.795417,426,84,91,219
23,gradient_boosting_default,5,0.50,0.782927,0.756967,0.747691,0.603226,0.772727,0.892157,0.812998,0.755956,455,55,123,187
24,gradient_boosting_f1_threshold,5,0.42,0.775610,0.755717,0.750664,0.648387,0.728261,0.852941,0.812998,0.755956,435,75,109,201


## Resultaten tegenover RF300
Standaardlabels en een afgestemde macro-F1-beslisregel staan apart. Die drempel is niet op buitenvalidatie gekozen. Grotere recall heeft alleen betekenis samen met precision, FN en FP.

In [3]:
display(summary[summary.model.eq('rf300_reference_default') | summary.model.str.startswith(tuple(families))][['model','macro_f1_mean','macro_f1_sd','recall_p_mean','precision_p_mean','fn_total','fp_total']])


,model,macro_f1_mean,macro_f1_sd,recall_p_mean,precision_p_mean,fn_total,fp_total
0,rf300_reference_default,0.761876,0.021452,0.604390,0.786176,614,256
3,gradient_boosting_default,0.765045,0.018125,0.612142,0.786357,602,258
4,gradient_boosting_f1_threshold,0.761895,0.008312,0.644388,0.749368,552,338


## Opnieuw trainen
De opgeslagen training herhalen kan met onderstaande vlag. Standaard worden alleen de echte resultaten gelezen, zonder onnodig opnieuw te fitten.

In [4]:
RERUN = False
if RERUN:
    import joblib
    from improve import run
    with joblib.parallel_config(backend='threading'):
        run(families)
else:
    print('Rapportcontrole; geen nieuwe training gestart.')


Rapportcontrole; geen nieuwe training gestart.
